# Статистическая ассоциация свистов и молний

**Кратко.** На масштабе минут–часов вероятность свиста рядом с грозой примерно в **2.5–3.4 раза выше**, чем без неё. Эффект есть у **ka1, ka2 и ka4**, днём и ночью; у ka1/ka2 — ещё во всех сезонах и полушариях. Это **корреляция**, а не доказательство причинности: ночь и география сами поднимают и молнии, и свисты. Следующий шаг — тест «доза–эффект», где эти факторы убираются из сравнения.

## 1. Вопрос, на который отвечает блок

Каузальный тест (секунды) проверяет: *приходит ли свист через доли секунды после конкретного разряда?* Там значимого excess нет — на секундном масштабе «эта молния → этот свист» не видно.

Здесь другой вопрос:

> Если за последние минуты в окрестности подспутниковой точки была молния, **выше ли доля свистов**, чем в точках, где молний не было?

Это связь с **грозовой активностью** (минуты–часы, сотни–тысячи км), а не travel time одного разряда по силовой трубке.


## 2. Дизайн теста

### 2.1. Что берём за наблюдение

| Объект | Определение |
|---|---|
| **Сэмпл** | Одна секунда записи предсказаний: `timestamp` (UTC), `geo_lat`, `geo_lon`, `verdict` (0/1). Время в файлах исходно МСК → при загрузке −3 ч |
| **Подспутниковая точка (надир)** | Проекция аппарата на поверхность (`geo_lat`/`geo_lon`). Не координата антенны в космосе |
| **Молния** | Разряд WWLLN AE (UTC): время, широта, долгота; `energy_min=None` — все разряды |
| **Ночь** | `SZA > 98°` в точке надира — тень **D-области (~70–90 км)**. Не колонка `is_night` из файла (тень КА на ~827 км) |

Шаг `timestamp` — **1 с**. За сутки теоретически 86 400 сэмплов; медиана покрытия ~86 000.

### 2.2. Период и аппараты

- **ka1, ka2, ka4** считаются **отдельно**, каждый на **своих** сутках (`days_by_sc`). Пересечения покрытий не требуем: иначе ka4 (только август) схлопнул бы всех к 10 дням.
- Режим **`spread`**: сутки равномерно по покрытию данного КА.
  - **ka1 / ka2:** **21 сутки** на год (янв–окт 2025).
  - **ka4:** все доступные **~10 суток** (21–31.08.2025).
- Сезонный контроль — только **ka1∩ka2** на общих датах.

Почему не 21 день подряд в январе (старый pilot): соседние секунды одного витка — почти одно наблюдение. Независимая единица — **сутки**; равномерный разброс по году нужен, чтобы не смешивать «эффект молний» с «эффектом января».

### 2.3. Подвыборка внутри суток

Из ~86 000 секунд берём случайные **12 000** (~14%), seed фиксирован. Выборка **без** дотягивания лишних `verdict=1` — иначе завысили бы базовую долю свистов.

Больше дней важнее, чем больше точек в дне: с 6 000 до 24 000 оценка почти не двигается, смена суток — сильнее.

### 2.4. Окно поиска молний

Только **прошлое** (lookback): до **180 мин** назад, до **2000 км** от надира по поверхности (haversine). Будущее не берём — иначе «молния после свиста» попала бы в статистику ассоциации.

AE читаем за **день±1**: lookback 3 часа у сэмпла около полуночи уходит в соседние сутки.

Два способа свернуть молнии в одну величину на сэмпл:

1. **Сетка экспозиции** (R, Δt) → тепловые карты и lift (см. §2.6–2.7).
2. **Ближайшая / последняя молния** → кривые P vs расстояние и P vs время (§4, рис. 3).

### 2.5. Сетка порогов

```
R, км:     50, 100, 250, 500, 1000, 1500, 2000
Δt, мин:   1, 5, 15, 30, 60, 120, 180
```

Пороги **накопительные**, не независимые кольца: ячейка (1000 км, 30 мин) = «есть ≥1 разряд ближе 1000 км и не старше 30 мин». Поэтому числа экспозиций по строке/столбцу монотонно растут; сравнивать соседние ячейки как «кольцо 500–1000» нельзя без дополнительной разности.

**Опорные ячейки** (назначены **до** просмотра чисел, белые квадраты на картах):

| Ячейка | Смысл |
|---|---|
| **R = 100 км, Δt = 5 мин** | гроза почти под аппаратом и только что |
| **R = 1000 км, Δt = 30 мин** | масштаб грозового очага, много статистики |

Максимум lift по всей сетке **не берём**: это максимум из ~50 коррелированных оценок, смещён вверх и обычно садится в дальний угол (большие R и Δt), где «нет молний» ≈ приполярье / пустой океан.

### 2.6. Четыре счётчика (таблица 2×2 в каждой ячейке)

Для каждой ячейки сетки (R, Δt) по всем сэмплам накапливается таблица 2×2:

|  | `verdict=1` | `verdict=0` |
|---|---|---|
| **экспонирован** (есть молния ближе R и свежее Δt) | `n_exp_v1` | `n_exp_v0` |
| **не экспонирован** | `n_no_v1` | `n_no_v0` |

Сэмпл **экспонирован** в ячейке (R, Δt), если хоть один разряд был ближе R километров и свежее Δt минут.

Отсюда:

$$
P_{\mathrm{есть}} = P_{\mathrm{exp}} = \frac{n_{\mathrm{exp\_v1}}}{n_{\mathrm{exp\_v1}}+n_{\mathrm{exp\_v0}}},\qquad
P_{\mathrm{нет}} = P_{\mathrm{no}} = \frac{n_{\mathrm{no\_v1}}}{n_{\mathrm{no\_v1}}+n_{\mathrm{no\_v0}}}
$$

$$
\mathrm{lift} = \frac{P_{\mathrm{есть}}}{P_{\mathrm{нет}}}
$$

- `lift = 1` — свисты одинаково часты с молнией и без;
- `lift = 3` — рядом с молнией свист втрое вероятнее (relative risk двух частот, не модель uplift).

Ячейки, где в **какой-то** из групп меньше **200** наблюдений, дают **NaN** (белые клетки на картах), а не число: иначе шум при R = 50 км / Δt = 1 мин читался бы как «пик».

На тепловых картах ниже показываем **две вероятности** (общая шкала); lift — в таблицах.

### 2.7. Как строятся кривые P vs расстояние и P vs время

Это **другой** срез тех же сэмплов, не накопительная сетка lift.

**P vs расстояние.** Среди молний за последние 180 мин берём **минимальное** расстояние до надира → кладём в бин (0–50, 50–100, …, ≥2000 км) → в бине считаем долю `verdict=1`.

**P vs время.** Фиксируем радиус R. Среди молний ближе R берём возраст **самой свежей** (минуты: время сэмпла минус время молнии) → бин (0–1, 1–5, …, 120–180 мин) → снова доля `verdict=1`.  
Две кривые: R ≤ **2000** км и R ≤ **250** км.  
Если за 180 мин в радиусе R молний не было — сэмпл **не** на оси X (бинов «>180» нет); его доля идёт в горизонталь «нет в R км».

Возражения к обеим кривым одни и те же по сути: «близко / недавно» отбирает грозовые регионы и часто ночь/LST. Night/day и сезоны ослабляют отдельные отговорки; географию полностью убирает только доза–эффект.


## 3. Вопросы к дизайну

**Почему минуты–часы, если свист летит секунды?**  
Секундный каузальный тест уже сделан — excess ≈ 0. Этот блок отвечает на другой вопрос: связана ли доля свистов с **грозовой обстановкой** в окрестности, а не с задержкой одного разряда.

**Почему не «радиус силовой трубки» (десятки км)?**  
Блок про активность очага. 100–250 км ближе к масштабу трубки; 1000–2000 км — к очагу. Оба в сетке

**Почему haversine, а не наклонная дальность до антенны?**  
Нужна удалённость разряда от **надира по Земле**, не км по прямой до спутника на 827 км. Это географический прокси окрестности, не точный footpoint силовой линии (линии не вертикальны; IGRF не используется). Ошибка сферы vs эллипсоид ~0.5% при бинах от 50 км несущественна.

**Почему пик P на дневной карте иногда не в углу (R=50, Δt=1)?**  
Угол днём редко экспонирован (мало молний почти под треком) → шум или NaN. Сырой P\|lightning ещё тянет состав выборки; форму ассоциации надёжнее смотреть по lift и по опорным ячейкам с n ≫ 200.

**Почему ночь — SZA > 98°, а не `is_night`?**  
Нужна ночь **D-области** под аппаратом (поглощение ОНЧ). `is_night` в файле — тень **самого** КА на ~827 км (фактически SZA ≳ 117°). Это разные вещи.

**Что значат серые / коричневые линии на P vs время?**  
Сэмплы без молний в R ≤ 2000 км (серая) или R ≤ 250 км (коричневая) за всё окно. Отдельный контроль, часто приполярье / океан.

**Зачем общие и свои шкалы у карт P?**  
Общая: один цвет = одна вероятность, можно сравнить «есть» и «нет». Свои: видна форма каждой карты (иначе контроль «сплющивается»); уровни между панелями не сравнивать. В сводке свои шкалы — один раз, на ka1.

**Зачем фиксировать опорные ячейки, а не max lift?**  
Max по сетке смещён вверх и любит дальний угол с вырожденным контролем.


## 4. Результаты

Таблицы и рисунки сняты с более раннего прогона блока ассоциации, чем тот, что сейчас сохранён в `wwlln_reader.ipynb`. Lift там отличается во втором знаке (ka1, 1000 км / 30 мин: 3.43 здесь, 3.32 в `wwlln_reader`); выводы те же.

### 4.1. Основной прогон 

| КА | дни | R, км | Δt, мин | P(есть) | P(нет) | **lift** | n(есть) |
|---|---|---|---|---|---|---|---|
| ka1 | 21, spread по году | 100 | 5 | 0.257 | 0.080 | **3.23** | 2 692 |
| ka1 |  | 1000 | 30 | 0.165 | 0.048 | **3.43** | 72 409 |
| ka2 | 21, spread по своему покрытию | 100 | 5 | 0.301 | 0.093 | **3.24** | 2 616 |
| ka2 |  | 1000 | 30 | 0.187 | 0.059 | **3.20** | 71 179 |
| ka4 | 10, 21–31.08.2025 | 100 | 5 | 0.520 | 0.208 | **2.50** | 2 062 |
| ka4 |  | 1000 | 30 | 0.370 | 0.138 | **2.69** | 38 926 |

**Вывод.** У ka1 и ka2 рядом с грозой доля свистов стабильно в **~3.2–3.4 раза** выше фона — на близкой ячейке и на ячейке очага, с тысячами–десятками тысяч экспозиций. У ka4 абсолютные P выше (короткий «громкий» август), но **контраст** того же порядка (lift ≈ 2.5–2.7). Эффект не артефакт одного аппарата; ka4 нельзя читать как годовой аналог ka1/ka2 без оговорки про 10 дней.

![Тепловые карты ka1, общая шкала](stat_assoc_figs/ka1_heatmaps_shared.png)

*Рис. 1. ka1: P(свист \| молния) и P(свист \| нет молнии), **общая** цветовая шкала. Белые квадраты — опорные ячейки.*

![Тепловые карты ka1, свои шкалы](stat_assoc_figs/ka1_heatmaps_own.png)

*Рис. 2. ka1: те же две карты с **индивидуальными** шкалами (пример; у ka2/ka4 в сводке только общая шкала). Сравнима форма внутри панели, не абсолютный цвет между панелями.*

![P vs расстояние и время, ka1](stat_assoc_figs/ka1_P_vs_dist_time.png)

*Рис. 3. ka1: P падает с расстоянием до ближайшей молнии и с временем с последней. На панели времени — две кривые (R ≤ 2000 км и R ≤ 250 км) и горизонтали «нет молний» в этих радиусах.*

Те же картины у ka2 и ka4:

![Тепловые карты ka2](stat_assoc_figs/ka2_heatmaps_shared.png)

![P vs расстояние и время, ka2](stat_assoc_figs/ka2_P_vs_dist_time.png)

![Тепловые карты ka4](stat_assoc_figs/ka4_heatmaps_shared.png)

![P vs расстояние и время, ka4](stat_assoc_figs/ka4_P_vs_dist_time.png)


### 4.2. Контроль ночь / день

Ночь и день разделены по SZA > 98° (то же определение, что во всём проекте). Lift в опорных ячейках:

| КА | условие | R=100, Δt=5 | R=1000, Δt=30 |
|---|---|---|---|
| ka1 | ночь | 2.28 | 2.76 |
| ka1 | день | 4.06 | 4.12 |
| ka2 | ночь | 2.34 | 2.63 |
| ka2 | день | 5.10 | 3.64 |
| ka4 | ночь | 2.13 | 2.24 |
| ka4 | день | 2.91 | 2.92 |

**Вывод.** Lift > 2 во всех шести комбинациях КА × ночь/день. Днём абсолютные вероятности ниже, но **контраст не пропадает** (часто даже выше). Значит, корреляция не сводится к тому, что «и молнии, и свисты просто ночные».

![ka1 ночь, P vs dist/time](stat_assoc_figs/ka1_night_P_vs_dist_time.png)

![ka1 день, P vs dist/time](stat_assoc_figs/ka1_day_P_vs_dist_time.png)


### 4.3. Сезонный контроль (ka1∩ka2; 4 окна × 2 полушария)

Спутник за виток пересекает оба полушария; в июле на севере лето, на юге зима — без разделения «сезон» в выборке смешан. Считаем **раздельно по полушариям**, по 7 суток в окне; только ka1∩ka2 (у ka4 нет покрытия вне конца августа).

Lift в опорной ячейке (1000 км, 30 мин):

| окно | полушарие | локальный сезон | lift | P(база) |
|---|---|---|---|---|
| янв | N | зима | 2.38 | 0.089 |
| янв | S | лето | 4.29 | 0.119 |
| апр | N | весна | 2.73 | 0.134 |
| апр | S | осень | 3.27 | 0.114 |
| июл | N | лето | 2.98 | 0.088 |
| июл | S | зима | 2.13 | 0.021 |
| окт | N | осень | 3.91 | 0.087 |
| окт | S | весна | 3.20 | 0.075 |

**Вывод.** Во всех восьми комбинациях lift > 2. Абсолютный фон гуляет сильно (южная зима: P(база) ≈ 0.02), но **контраст «молния рядом / нет» держится** — это не артефакт одного сезона или одного полушария. `P(база)` в таблице как раз чтобы не путать одинаковый lift при разном фоне.

![Сезонные кривые](stat_assoc_figs/season_P_vs_dist_time.png)

*Рис. 4. P vs расстояние и vs время в четырёх сезонных окнах, N/S отдельно (кривая R ≤ 250 км в сезонном контроле не строится).*


## 5. Итог

- Доля свистов статистически связана с близостью грозовой активности на масштабе минут–часов и сотен–тысяч км.
- Связь воспроизводится у ka1, ka2 и (на коротком августе) ka4; у ka1/ka2 — днём/ночью и по сезонам/полушариям.
- Типичный порядок: lift ≈ 2.5–3.4 в заранее выбранных опорных ячейках.
- Часть lift-a может быть общей зависимостью от широты, LST и освещённости.

**Главная оговорка дизайна.** Контроль «нет молний в R» при больших R смещается в приполярье и пустой океан. Опорные ячейки с умеренным R/Δt страдают меньше, но географию и LST полностью не убирают — отсюда следующий блок.


## 6. Переход к тесту «доза–эффект»

Статистическая ассоциация отвечает: *есть ли корреляция?* — да, устойчивая, порядка трёх.

Следующий вопрос:

> При **тех же** широте, местном времени, сезоне и освещённости — растёт ли вероятность свиста, когда в окрестности больше молний?

Блок «доза–эффект»:

1. Сетка `(время 5 мин × широта 2° × долгота 2°)` вдоль трека.
2. **Доза** — число молний в радиусе ~1000 км за ~30 мин.
3. Сравнение внутри страт `широта × LST × сезон × ночь/день × КА` (Мантель–Хензель).
4. Нуль: та же точка, доза с других суток.

Если рост сохранится — молнии несут информацию сверх координат и времени. Если исчезнет — корреляция этого блока была в основном конфаундером.
